RNN - Erro dos pesos computados e usado somente durante a iteração

In [1]:
import numpy as np
from numpy import linalg as LA
import pandas as pd
import operator as op
import ipynbname
import math
import matplotlib.cm as cm
import optuna
from optuna.samplers import RandomSampler
from optuna.samplers import TPESampler
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_pareto_front
from optuna.importance import get_param_importances
from optuna.exceptions import TrialPruned
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import matplotlib as mpl
#from Testing.RTLO import *
from Functions.RLS import *
from Functions.Utils_RTLO import *
from Functions.Graphs import *
from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.metrics import mean_squared_error as MSE

FileName = ipynbname.name()

params = [14, 12, 2, 0.01, 1e-08, 0.0001, 2]
df = pd.read_csv(r'Dataset\Bearing1_1.csv')
sig = df['PC1'].values

def PlotPredError(rtlo,w=9,h=3):
    s = len(rtlo.yWAPE)
    t = rtlo.t
    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(w, h))
    axes = axes.flatten()
    ax1,ax2,ax3,ax4 = axes[0], axes[1], axes[2], axes[3]

    ax1.plot(t, rtlo.yR, color='black',label='Y-Real', linestyle='-')
    ax1.plot(t, rtlo.yP, color='blue',label='Y-Pred', linestyle='-')
    ax1.plot(t, rtlo.yL, color='blue', linestyle='--')
    ax1.plot(t, rtlo.yU, color='blue', linestyle='--')
    
    ax1.set_title('Y - Real x Prediction')
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y', color='black')
    ax1.legend()

    ax2.plot(t, rtlo.eR, color='black',label='e-Real', linestyle='-')
    ax2.plot(t, rtlo.eP, color='blue',label='e-Pred', linestyle='-')
    ax2.set_title('Error - Real x Prediction')
    ax2.set_xlabel('X')
    ax2.set_ylabel('Prediction Error', color='black') 
    ax2.legend()
    
    ax3.plot(t[-s:], rtlo.yWAPE, color='blue',label='WAPE', linestyle='-')
    ax3.set_title('Prediction WAPE')
    ax3.set_xlabel('X')
    ax3.set_ylabel('WAPE', color='black') 

    '''ax4.plot(t[-s:], rtlo.rWAPE, color='blue',label='WAPE', linestyle='-')
    ax4.set_title('RUL Prediction WAPE')
    ax4.set_xlabel('X')
    ax4.set_ylabel('WAPE', color='black') '''

    fig.tight_layout()  # otherwise the right y-label is slightly clipped
    plt.show()

def PlotPredErrorPLY(rtlo, w=800, h=300):
    # s: tamanho do vetor WAPE (caso comece depois do início)
    t = rtlo.t
    
    # Criando o layout de 1 linha e 4 colunas
    fig = make_subplots(
        rows=1, cols=3, 
        shared_xaxes=True,
        subplot_titles=('Y - Real x Prediction', 'RUL - Real x Pred', 'Error - Real x Pred', 'Prediction WAPE', 'RUL Prediction WAPE')
    )

    # --- Subplot 1: Y Real x Pred (com Intervalos) ---
    fig.add_trace(go.Scatter(x=t, y=rtlo.yR, name='Y-Real', line=dict(color='black')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=rtlo.yP, name='Y-Pred', line=dict(color='blue')), row=1, col=1)
    # Intervalos (Dashed)
    fig.add_trace(go.Scatter(x=t, y=rtlo.yL, name='y-Lower', line=dict(color='blue', dash='dash'), showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=rtlo.yU, name='y-Upper', line=dict(color='blue', dash='dash'), showlegend=False), row=1, col=1)

    fig.add_trace(go.Scatter(x=t, y=rtlo.rulR, name='rul R', line=dict(color='black'), showlegend=True), row=1, col=2)
    fig.add_trace(go.Scatter(x=t, y=rtlo.rulP, name='rul P', line=dict(color='blue'), showlegend=True), row=1, col=2)
    fig.add_trace(go.Scatter(x=t, y=rtlo.rulL, name='rul L', line=dict(color='red'), showlegend=True), row=1, col=2)
    fig.add_trace(go.Scatter(x=t, y=rtlo.rulU, name='rul U', line=dict(color='green'), showlegend=True), row=1, col=2)

    #fig.add_trace(go.Scatter(x=t, y=rtlo.εM_hist, name='rul R', line=dict(color='black'), showlegend=False), row=1, col=3)
    fig.add_trace(go.Scatter(x=t, y=rtlo.eR, name='erro R', line=dict(color='black')), row=1, col=3)
    fig.add_trace(go.Scatter(x=t, y=rtlo.eP, name='erro P', line=dict(color='blue')), row=1, col=3)

    # Atualizando Layout e Eixos
    fig.update_layout(
        width=w, height=h,
        title_text=f"RTLO Model Performance Analysis",
        template='plotly_white',
        showlegend=True,
        margin=dict(l=40, r=40, t=80, b=40)
    )

    # Labels dos eixos (opcional, já que os títulos ajudam)
    fig.update_xaxes(title_text="Time / Index")
    fig.update_yaxes(title_text="Amplitude", col=1)
    fig.update_yaxes(title_text="Error", col=2)

    fig.show()
    
def SelSampler(mode='auto'):
    '''mode: auto, random,  tpe'''
    if mode == 'auto':
        sampler = None
    elif mode == 'tpe':
        sampler = optuna.samplers.TPESampler(multivariate=True, constant_linker=True,group=True,n_startup_trials=2000)
    elif mode == 'random':
        sampler=RandomSampler()
    return sampler

c:\Users\Usuário\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [553]:
class RTLO:
    def __init__(self, nI,nR,nO,ηS=[0.1,0.1,0.1], τ=10,lr=1e-5):
        np.random.seed(42)
        self.k = 1
        self.j = nI-1
        self.t = np.array([])
        self.ref = None
        self.act = 'tanh'
        self.nI, self.nR, self.nO = nI, nR, nO

        self.ηS = np.array(ηS)
        self.τ = τ
        self.ρ = 0.01

        self.xPi = np.zeros(nI)
        self.hP, self.hU, self.hL = [np.zeros(nR) for i in range(3)]

        self.pS = np.zeros((self.nR, self.nR))
        self.qS = np.zeros((self.nR, self.nI))

        self.ΔOS = np.zeros((nO, nR))
        self.ΔRS = np.zeros((nR, nR))
        self.ΔIS = np.zeros((nR, nI))
        
        self.wI = XavierUniform([nR, nI],sd=42)
        self.wR = XavierUniform([nR, nR],sd=41)
        self.wO = XavierUniform([nO, nR],sd=40)
        self.BS = XavierUniform([nR, nO],sd=39)

        self.rls = RLS_LogarithmicRegressor(0.95,1e7)
        
        self.yP, self.yR, self.yL, self.yU = [np.array([]) for i in range(4)]
        self.yP_hist = np.zeros(self.nI)
        self.eS = np.zeros(nI)
        self.eS2 = np.zeros(nI)
        self.eP = np.array([])
        self.eR = np.array([])

        self.εY, self.εM, self.εE, self.εR, self.ΣW = [0 for i in range(5)]
        self.εM_hist = np.array([])
        self.εR_hist = np.array([])

        self.μrWAPE = 0
        self.MPsum = 0

        self.rR = 1e-9
        self.rP = 1e-10
        self.rL = 1e-11
        self.rU = 1e-12
        self.rRsum = 0

        self.rulR, self.rulP, self.rulL, self.rulU = [np.array([]) for i in range(4)]


    def PredSingle(self,x):

        u = np.dot(self.wR, self.hS) + np.dot(self.wI, x)
        h = self.hS + (-self.hS + Activation(u,self.act))/self.τ
        y = np.dot(self.wO, h)

        return y

    def fit(self,xP,yR,store=False,show=False):

        exp=2
        W = self.j**exp
        η1,η2,η3 = self.ηS        
        #print('fit self.hP:',self.hP[-5:])
        uS = self.wR @ self.hP + self.wI @ xP
        #hP = self.hP + (-self.hP + Activation(uS,self.act))/self.τ
        hP = self.hP*(1-1/self.τ) +Activation(uS,self.act)/self.τ
        yP = self.wO @ hP
        eS = yR-yP

        if show: 
            #print('yR:',yR)
            #print('xP:',xP)
            print('fit  yP:',yP[0])
            #print('fit  hP:',self.hP)
            #print('fit  hP2:',hP)
            #print('fit xP:',xP)
            #print('fit uP:',uS)
            #print('fit  wR:\n',self.wR)
            #print('fit  wI\n:',self.wI)
            #print('fit  wO\n:',self.wO)

        self.pS = np.outer(dActivation(uS,self.act),self.hP)/self.τ + (1-1/self.τ)*self.pS
        self.qS = np.outer(dActivation(uS,self.act),self.xPi)/self.τ + (1-1/self.τ)*self.qS

        δOS = η1*np.outer(eS,hP)
        δRS = η2*np.outer((self.BS@eS),np.ones(self.nR))*self.pS
        δIS = η3*np.outer(np.dot(self.BS, eS),np.ones(self.nI))*self.qS

        self.ΔIS = (self.ΔIS*(self.k-1) + δIS)/self.k
        self.ΔRS = (self.ΔRS*(self.k-1) + δRS)/self.k
        self.ΔOS = (self.ΔOS*(self.k-1) + δOS)/self.k

        self.wI = self.wI + δIS
        self.wR = self.wR + δRS
        self.wO = self.wO + δOS

        self.hP = hP
        self.xPi = xP

        self.UpdateRLS(yP,yR)

        ΣW = self.ΣW + W
        ΔY = np.abs((yR-yP)/yR)
        ΔM = np.linalg.norm(ΔY,ord=2)
        ΔR = np.abs((self.rR-self.rP)/(self.rR+1e-9))
        ΔE = np.abs((self.eR[-1]-self.eP[-1])/self.eR[-1])

        self.εY = ((self.εY*self.ΣW) + (W*ΔY[0]))/ΣW
        self.εM = ((self.εM*self.ΣW) + (W*ΔM))/ΣW
        self.εR = ((self.εR*self.ΣW) + (W*ΔR))/ΣW
        self.εE = ((self.εE*self.ΣW) + (W*ΔE))/ΣW
        self.ΣW = ΣW

        if store:
            self.yR = np.append(self.yR,yR[0])
            self.εM_hist = np.append(self.εM_hist,self.εM)
            #self.εR_hist = np.append(self.εR_hist,self.εR)

        self.k = self.k+1
        self.j = self.j+1
        self.t = np.append(self.t,self.k + self.nI)
        self.yP_hist = np.delete(np.append(self.yP_hist,yP[0]),0)
        #self.ηS = self.ηS/(1 + self.decay*self.k)


    def PredRulIntr(self, x,lim=0.2,maxRul=110,store=False,show=False):
        #print('yH:',self.yP_hist)
        #print('eS: ',self.eS)

        for i,y in enumerate(self.yP_hist):
            if y != 0:
                self.eS2[i] = self.rls.predict(np.abs(y))
        #print('eS2:',self.eS2)

        xP,xL,xU =x.copy(), (x-(self.ρ*self.eS2)).copy(),(x+(self.ρ*self.eS2)).copy()    

        #xP,xL,xU =x.copy(), x.copy()*0.999, x.copy()*1.00

        predict = True
        PredRuls = [True for i in range(3)]
        PredVals, Ruls = [0 for i in range(3)], [0 for i in range(3)]
        wR,wI,wO = self.wR,self.wI,self.wO

        wRp, wRn = np.maximum(0, self.wR), np.abs(np.minimum(0, self.wR))
        wIp, wIn = np.maximum(0, self.wI), np.abs(np.minimum(0, self.wI))
        wOp, wOn = np.maximum(0, self.wO), np.abs(np.minimum(0, self.wO))
        #hU,hL = np.maximum(0, self.hS), np.minimum(0, self.hS)
        hL,hP,hU = [self.hP.copy() for i in range(3)]

        while predict:
            uP = wR@hP + wI@xP
            uL = (wRp @ hL - wRn @ hU) + (wIp @ xL - wIn @ xU)
            uU = (wRp @ hU - wRn @ hL) + (wIp @ xU - wIn @ xL)

            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            hL = hL*(1-1/self.τ) + Activation(uL,self.act)/self.τ
            hU = hU*(1-1/self.τ) + Activation(uU,self.act)/self.τ
    
            yP = (wO@hP)
            yL = (wOp @ hL - wOn @ hU)
            yU = (wOp @ hU - wOn @ hL)

            #if show: print(yP)
            yP = yP[0]
            yL = yL[0]
            yU = yU[0]

            xP = np.delete(np.append(xP,yP),0)
            xL = np.delete(np.append(xL,yL),0)
            xU = np.delete(np.append(xU,yU),0)
            PredVals = [yL,yP,yU]

            if Ruls[0] == 0:
                self.yL = np.append(self.yL,yL)
                self.yP = np.append(self.yP,yP)
                self.yU = np.append(self.yU,yU)
            
            CheckPred,CheckLim=0,0
            for i in range(3):
                if PredRuls[i]: 
                    Ruls[i] = Ruls[i]+1
                    if Ruls[i] >= maxRul:
                        CheckLim = CheckLim + 1
                        Ruls[i] = maxRul
                        PredRuls[i] = False
                if PredVals[i] < lim: PredRuls[i] = False
                if not PredRuls[i]: CheckPred = CheckPred + 1
            if CheckPred == 3:break
            if CheckLim == 3:break
        
        self.rR=self.ref-self.k
        self.rL,self.rP,self.rU = Ruls

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulL = np.append(self.rulL,self.rL)
            self.rulP = np.append(self.rulP,self.rP)
            self.rulU = np.append(self.rulU,self.rU)
    
    def PredRulIntr2(self, x,lim=0.2,maxRul=110,store=False,show=False):
        #print('yH:',self.yP_hist)
        #print('eS: ',self.eS)

        for i,y in enumerate(self.yP_hist):
            if y != 0:
                self.eS2[i] = self.rls.predict(np.abs(y))
        #print('eS2:',self.eS2)

        xP,xL,xU =x.copy(), (x-(self.ρ*self.eS2)).copy(),(x+(self.ρ*self.eS2)).copy()    

        #xP,xL,xU =x.copy(), x.copy()*0.999, x.copy()*1.00

        predict = True
        PredRuls = [True for i in range(3)]
        PredVals, Ruls = [0 for i in range(3)], [0 for i in range(3)]
        wR,wI,wO = self.wR,self.wI,self.wO
        ep = 0.05
        wRp, wRn = np.maximum((1+ep)*wR, wR/(1+ep)), np.minimum((1+ep)*wR, wR/(1+ep))
        wIp, wIn = np.maximum((1+ep)*wI, wI/(1+ep)), np.minimum((1+ep)*wI, wI/(1+ep))
        wOp, wOn = np.maximum((1+ep)*wO, wO/(1+ep)), np.minimum((1+ep)*wO, wO/(1+ep))

        hP = self.hP.copy()  
        hU, hL = np.maximum((1+ep)*hP, hP/(1+ep)), np.minimum((1+ep)*hP, hP/(1+ep))

        print('wRn',wRn[0][:5])
        print('wR ',wR[0][:5])
        print('wRp',wRp[0][:5])
        print('---')


        while predict:
            uP = wR@hP + wI@xP
            uL = (wRn @ hL) + (wIn @ xL)
            uU = (wRp @ hU) + (wIp @ xU)

            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            hL = hL*(1-1/self.τ) + Activation(uL,self.act)/self.τ
            hU = hU*(1-1/self.τ) + Activation(uU,self.act)/self.τ
    
            yP = (wO@hP)[0]
            yL = (wOn @ hL)[0]
            yU = (wOp @ hU)[0]

            PredVals = sorted([yL,yP,yU])

            yL,yP,yU = PredVals

            xP = np.delete(np.append(xP,yP),0)
            xL = np.delete(np.append(xL,yL),0)
            xU = np.delete(np.append(xU,yU),0)

            if Ruls[0] == 0:
                self.yL = np.append(self.yL,yL)
                self.yP = np.append(self.yP,yP)
                self.yU = np.append(self.yU,yU)
            
            CheckPred,CheckLim=0,0
            for i in range(3):
                if PredRuls[i]: 
                    Ruls[i] = Ruls[i]+1
                    if Ruls[i] >= maxRul:
                        CheckLim = CheckLim + 1
                        Ruls[i] = maxRul
                        PredRuls[i] = False
                if PredVals[i] < lim: PredRuls[i] = False
                if not PredRuls[i]: CheckPred = CheckPred + 1
            if CheckPred == 3:break
            if CheckLim == 3:break
        
        self.rR=self.ref-self.k
        self.rL,self.rP,self.rU = Ruls

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulL = np.append(self.rulL,self.rL)
            self.rulP = np.append(self.rulP,self.rP)
            self.rulU = np.append(self.rulU,self.rU)
    
    def PredRul(self, x,lim=0.2,store=False):
        xP = x.copy()
        rulP=0
        predict = True
        hP = self.hP.copy()
        while predict:
            uP = (self.wR @ hP) + (self.wI @ xP)
            hP = hP*(1-1/self.τ) + Activation(uP,self.act)/self.τ
            yP = (self.wO @ hP)[0]
            xP = np.delete(np.append(xP,yP),0)
            if store:
                if rulP==0:
                    self.yP = np.append(self.yP,yP)

            if predict: rulP = rulP+1
            if yP < lim: predict = False
            if rulP >= 110:
                break

        self.rR=self.ref-self.k
        self.rP = rulP

        if store:
            self.rulR = np.append(self.rulR,self.rR)
            self.rulP = np.append(self.rulP,self.rP)
            
    
    def UpdateRLS(self,yP,yR):
        eP = np.abs(self.rls.predict(np.abs(yP[0])))
        eR = np.abs(yP-yR)[0]
        self.rls.update(np.abs(yP[0]), eR)
        self.eR = np.append(self.eR,eR)
        self.eP = np.append(self.eP,eP)
        self.eS = np.append(self.eS,eP)
        self.eS = np.delete(self.eS,0)


#Optimize parameters for minimize error of degradation prediction

In [ ]:
rates = [1/(10**i) for i in range(1,7)][::-1]
def objective(trial):

    nI = trial.suggest_int('nI', 8, 19) 
    nR = trial.suggest_int('nR', 30, 43) 
    nO = trial.suggest_int('nO', 1, 10) 
    N1 = trial.suggest_categorical('N1', rates[2:]) 
    N2 = trial.suggest_categorical('N2', rates[2:]) 
    N3 = trial.suggest_categorical('N3', rates) 
    τ = trial.suggest_int('τ', 1, 15)        
    X,Y = PrepareDataAhead(sig,n=nI,m=nO)
    rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ)
    rnn.ref = len(sig)-nI

    for i,_ in enumerate(X):
        rnn.fit(X[i],Y[i],store=True)

        if i == 50:
            if np.mean(rnn.εM_hist)>1:
                raise TrialPruned()

        if i % 20 == 0:  # Report every 10 time steps
            trial.report(rnn.εM, step=i)
            if trial.should_prune():
                raise optuna.TrialPruned()
            
    #return rnn.εY
    return rnn.εM

#pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
pruner=optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    direction="minimize",
    sampler=SelSampler(mode='auto'),
    pruner=pruner,
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}',
    load_if_exists=True)
study.optimize(objective, n_trials=4000)
params = list(study.best_params.values())
print('Erro:', study.best_value, 'parameters: ', params)

In [123]:
rates = [1/(10**i) for i in range(1,7)][::-1]
def objective(trial):

    nI = trial.suggest_int('nI', 2, 25) 
    nR = trial.suggest_int('nR', 1, 50) 
    nO = trial.suggest_int('nO', 1, 25) 
    N1 = trial.suggest_categorical('N1', rates[:]) 
    N2 = trial.suggest_categorical('N2', rates[:]) 
    N3 = trial.suggest_categorical('N3', rates) 
    τ = trial.suggest_int('τ', 1, 25)        
    X,Y = PrepareDataAhead(sig,n=nI,m=nO)
    rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ)
    rnn.ref = len(sig)-nI

    for i,_ in enumerate(X):
        rnn.PredRul(X[i],store=True)
        rnn.fit(X[i],Y[i],store=True)

        if i == 50:
            if np.mean(rnn.rulP)<10:
                raise TrialPruned()

        '''if i == 50:
            if np.mean(rnn.εM_hist)>1:
                raise TrialPruned()

        if i % 20 == 0:  # Report every 10 time steps
            trial.report(rnn.εM, step=i)
            if trial.should_prune():
                raise optuna.TrialPruned()'''
            
    #return rnn.εY
    return rnn.εM, rnn.εR

#pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
pruner=optuna.pruners.HyperbandPruner()

study = optuna.create_study(
    directions=["minimize", "minimize"],
    sampler=SelSampler(mode='auto'),
    pruner=pruner,
    #storage="sqlite:///" + f'Optuna/{FileName}_Prdct.db', study_name=f'P{4}',
    load_if_exists=True)
study.optimize(objective, n_trials=1000)

best_trials = study.best_trials

print(f"Encontrados {len(best_trials)} modelos na Fronteira de Pareto:")

for i, trial in enumerate(best_trials):
    print(f"\n--- Modelo Elite {i+1} ---")
    print(f"Valores: Erro_M = {trial.values[0]:.6f}, Erro_R = {trial.values[1]:.6f}")
    print(f"Parâmetros: {trial.params}")

# Se você quiser apenas os parâmetros do PRIMEIRO modelo da fronteira para testar:
first_best_params = best_trials[0].params


[I 2026-04-16 18:18:16,200] A new study created in memory with name: no-name-6db76fcd-3e0a-49c8-be70-2db29a40537e
[I 2026-04-16 18:18:16,300] Trial 0 finished with values: [0.48937432752282833, 0.745148506041204] and parameters: {'nI': 3, 'nR': 36, 'nO': 5, 'N1': 0.01, 'N2': 0.01, 'N3': 1e-06, 'τ': 4}.
[I 2026-04-16 18:18:16,396] Trial 1 finished with values: [1.1853026481877644, 1.7252752681070187] and parameters: {'nI': 18, 'nR': 23, 'nO': 24, 'N1': 0.01, 'N2': 0.01, 'N3': 0.01, 'τ': 12}.
[I 2026-04-16 18:18:16,502] Trial 2 finished with values: [1.200915267405982, 0.6149332314495388] and parameters: {'nI': 4, 'nR': 20, 'nO': 14, 'N1': 0.01, 'N2': 0.01, 'N3': 0.001, 'τ': 16}.
[I 2026-04-16 18:18:16,599] Trial 3 finished with values: [6.879289350496917, 1.6642323758389879] and parameters: {'nI': 5, 'nR': 31, 'nO': 25, 'N1': 1e-06, 'N2': 1e-05, 'N3': 1e-06, 'τ': 21}.
[I 2026-04-16 18:18:16,609] Trial 4 pruned. 
[I 2026-04-16 18:18:16,617] Trial 5 pruned. 
[I 2026-04-16 18:18:16,624] Tr

Encontrados 20 modelos na Fronteira de Pareto:

--- Modelo Elite 1 ---
Valores: Erro_M = 0.063743, Erro_R = 0.189189
Parâmetros: {'nI': 6, 'nR': 43, 'nO': 6, 'N1': 0.1, 'N2': 0.1, 'N3': 0.0001, 'τ': 16}

--- Modelo Elite 2 ---
Valores: Erro_M = 0.020650, Erro_R = 0.278211
Parâmetros: {'nI': 15, 'nR': 37, 'nO': 2, 'N1': 0.1, 'N2': 0.0001, 'N3': 0.1, 'τ': 18}

--- Modelo Elite 3 ---
Valores: Erro_M = 0.008096, Erro_R = 24492698.294337
Parâmetros: {'nI': 8, 'nR': 37, 'nO': 1, 'N1': 0.1, 'N2': 0.001, 'N3': 1e-06, 'τ': 12}

--- Modelo Elite 4 ---
Valores: Erro_M = 0.011598, Erro_R = 24490198.937391
Parâmetros: {'nI': 6, 'nR': 43, 'nO': 1, 'N1': 0.1, 'N2': 0.0001, 'N3': 0.1, 'τ': 19}

--- Modelo Elite 5 ---
Valores: Erro_M = 0.011372, Erro_R = 24490198.945324
Parâmetros: {'nI': 6, 'nR': 43, 'nO': 1, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.1, 'τ': 18}

--- Modelo Elite 6 ---
Valores: Erro_M = 0.023661, Erro_R = 0.245239
Parâmetros: {'nI': 12, 'nR': 26, 'nO': 2, 'N1': 0.1, 'N2': 0.1, 'N3': 0.0001, 'τ'

Valores: Erro_M = 0.023125, Erro_R = 0.408742
Parâmetros: {'nI': 15, 'nR': 41, 'nO': 2, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.001, 'τ': 1}

In [462]:
params = {'nI': 6, 'nR': 43, 'nO': 1, 'N1': 0.1, 'N2': 1e-05, 'N3': 0.1, 'τ': 18}
params = trial.params
params={'nI': 6, 'nR': 43, 'nO': 6, 'N1': 0.1, 'N2': 0.1, 'N3': 0.0001, 'τ': 16}
params = list(params.values())
#params = [8, 41, 1, 0.1, 0.01, 0.001, 2]


In [554]:
nI,nR,nO,N1,N2,N3,τ= params
X,Y = PrepareDataAhead(sig,n=nI,m=nO)
rnn = RTLO(nI,nR,nO,[N1,N2,N3],τ)
rnn.ref = len(sig)-nI
for i in range(len(X[:])):
    #rnn.PredRul(x=X[i],store=True)
    rnn.PredRulIntr2(x=X[i],maxRul=200,store=True,show=False)
    rnn.fit(X[i],Y[i],store=True,show=False)
#print(.wR[0])
print(rnn.εM,rnn.εR)  
PlotPredErrorPLY(rnn)


wRn [-0.1381587  -0.25177342  0.08895887 -0.2532302  -0.21276366]
wR  [-0.13157971 -0.23978421  0.09340681 -0.24117162 -0.20263206]
wRp [-0.12531401 -0.22836591  0.09807715 -0.22968726 -0.19298291]
---
wRn [-0.1381587  -0.25177342  0.08895887 -0.2532302  -0.21276366]
wR  [-0.13157971 -0.23978421  0.09340681 -0.24117162 -0.20263206]
wRp [-0.12531401 -0.22836591  0.09807715 -0.22968726 -0.19298291]
---
wRn [-0.1381605  -0.25175818  0.0889145  -0.25328482 -0.21279096]
wR  [-0.13158142 -0.23976969  0.09336022 -0.24122364 -0.20265805]
wRp [-0.12531564 -0.22835209  0.09802823 -0.2297368  -0.19300767]
---
wRn [-0.13816459 -0.25171535  0.08878592 -0.25344121 -0.2128641 ]
wR  [-0.13158533 -0.23972891  0.09322521 -0.24137258 -0.20272771]
wRp [-0.12531936 -0.22831324  0.09788648 -0.22987865 -0.19307401]
---
wRn [-0.13817043 -0.25163616  0.0885413  -0.25373538 -0.21299255]
wR  [-0.13159089 -0.23965349  0.09296836 -0.24165274 -0.20285005]
wRp [-0.12532465 -0.22824142  0.09761678 -0.23014547 -0.1931

In [26]:
i=50
r_m = np.mean(rnn.rulR[:i])
r_mL = np.mean(rnn.rulR[:i])*0.3
r_mU = np.mean(rnn.rulR[:i])*1.15
p_m = (np.mean(rnn.rulP[:i]))

print('lower:',r_mL,'mid:',r_m,'upper:',r_mU)
print('pred:',p_m)

i=50
f=90
r_m = np.mean(rnn.rulR[i:f])
r_mL = np.mean(rnn.rulR[i:f])*0.3
r_mU = np.mean(rnn.rulR[i:f])*1.5
p_m = (np.mean(rnn.rulP[i:f]))

print('lower:',r_mL,'mid:',r_m,'upper:',r_mU)
print('pred:',p_m)

lower: 26.849999999999998 mid: 89.5 upper: 102.925
pred: 0.16
lower: 13.35 mid: 44.5 upper: 66.75
pred: 0.0


In [486]:
#params =  [14, 8, 14, 0.001, 0.01, 1e-06, 1e-07, 29]
nI,nR,nO,N1,N2,N3,τ= params
ηS = [N1,N2,N3]
X,Y = PrepareDataAhead(sig,n=nI,m=nO)
rnn = RTLO(nI,nR,nO,ηS,τ)
rnn.ref = len(sig)-nI

i=0

In [488]:
#rnn.PredRul(x=X[i],store=True)
print('iteration:',i)
print('xP:',X[i])
print('yR:',Y[i])
rnn.PredRulIntr2(x=X[i],store=True,show=True)
rnn.fit(X[i],Y[i],store=True,show=False)
i=i+1

iteration: 1
xP: [0.91429178 0.91121962 0.90817817 0.90519636 0.90227154 0.89941598]
yR: [0.89663373 0.8939275  0.89128272 0.88870303 0.88617045 0.88366572]
[np.float64(-0.03730620483456837), np.float64(-0.04602629367076666), np.float64(-0.05561926571854693)]


In [496]:
v = [np.float64(-0.03730620483456837), np.float64(-0.04602629367076666), np.float64(-0.05561926571854693)]
v = sorted(v)
print(v)

[np.float64(-0.05561926571854693), np.float64(-0.04602629367076666), np.float64(-0.03730620483456837)]
